# Medical RAG - Retrieval-Augmented Generation over a Trusted Medical Corpus

**Task 2 - Standalone Medical RAG Application**

> **Educational demonstration only.** This notebook is not a clinical decision-support
> system and must not be used to diagnose or treat anyone. Always verify medical
> information with a qualified professional and the primary source documents.

---

## Section 1 - Introduction

### What is RAG?

A large language model only knows what was in its training data. It cannot cite a
source, it cannot be updated without retraining, and when it does not know something
it tends to produce fluent but invented text.

**Retrieval-Augmented Generation** fixes this by splitting the job in two:

1. **Retrieval** - search a trusted document collection for the passages most
   relevant to the question.
2. **Generation** - hand only those passages to the language model and instruct it
   to answer from them alone.

The model stops being the source of facts and becomes a reader and summariser of
facts we chose and can point to.

### Why RAG for medical information?

- **Traceability.** Every answer can be traced to a named guideline, so a reader can
  check it.
- **Grounding.** A wrong drug dose is not a harmless error. Restricting the model to
  retrieved text sharply narrows the room for invention.
- **Updatability.** Guidelines change. Replacing a document in the corpus updates the
  system's knowledge - no retraining involved.
- **Honest gaps.** When the corpus does not cover a question, the system is instructed
  to say so rather than guess.

### Objective

Build the complete core RAG pipeline, end to end, using only open-source components
and no paid APIs:

```
16 medical documents
        v  cleaning
        v  paragraph-based chunking
        v  open-source embeddings  (BAAI/bge-small-en-v1.5)
        v  FAISS exact vector search
        v  top-3 retrieval
        v  grounded prompt
        v  open-source instruction LLM  (Qwen2.5-1.5B-Instruct)
   answer based only on the retrieved context
```

Every component below is there because the task needs it. Nothing is added to look
sophisticated - Section 14 justifies each piece one by one.

> **Runtime:** CPU works. A GPU runtime (Runtime > Change runtime type > T4 GPU) makes
> Section 9 onwards noticeably faster.

---
## Section 2 - Install Dependencies

Four libraries, one job each. No API keys, no accounts, nothing paid.

| Library | Why it is here |
|---|---|
| `sentence-transformers` | loads the open-source embedding model |
| `faiss-cpu` | stores the embeddings and does the similarity search |
| `transformers` | loads the open-source instruction LLM |
| `torch` | the tensor backend both models run on (preinstalled in Colab) |

In [6]:
%pip install -q "sentence-transformers>=3.0" "faiss-cpu>=1.8" "transformers>=4.57" "torch>=2.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 90.3 MB/s eta 0:00:00


---
## Section 3 - Imports and Configuration

Every tunable value lives in this one cell, so nothing important is buried further
down. The reasoning behind each number is given in the section that uses it.

In [7]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

# ---------------------------------------------------------------------------
# Configuration. Everything tunable lives here so no value is buried in code.
# ---------------------------------------------------------------------------
CORPUS_DIR = "corpus"

# Chunking: pack whole paragraphs together until the chunk reaches ~180 words,
# then start the next chunk by repeating the last paragraph. See Section 5.
MAX_CHUNK_WORDS = 180
OVERLAP_PARAGRAPHS = 1

# Embeddings: 33M parameters, 384 dimensions, strong on MTEB retrieval, CPU-friendly.
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
# BGE models are trained asymmetrically: queries get this prefix, passages get none.
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

# Retrieval: the assignment asks for the top 3 relevant documents.
TOP_K = 3

# Generation: open weights (Apache-2.0), small enough for a free Colab runtime.
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 320

SEED = 42

---
## Section 4 - Medical Knowledge Base

16 reference documents (the task asks for 10-20), each summarising the diagnosis and
management of one condition, written from published guidance by recognised bodies:
**WHO**, **CDC**, **NICE**, and specialist societies (ACC/AHA, ADA, GINA, GOLD,
ATS/IDSA, KDIGO, ESC, Surviving Sepsis Campaign, World Allergy Organization,
American Thyroid Association, BSH).

Each document carries the metadata the task asks for - `id`, `title`, `source`,
`category` - in a small `---` block at the top of the file, followed by the text.

Coverage: cardiovascular (hypertension, acute coronary syndrome), endocrine (type 2
diabetes, hypothyroidism), respiratory (asthma, COPD), infectious disease (pneumonia,
malaria, tuberculosis, UTI), emergency and critical care (stroke, sepsis, anaphylaxis),
nephrology (chronic kidney disease), haematology (iron deficiency anaemia), and public
health (antimicrobial resistance).

> **On provenance:** these are condensed summaries written for this demonstration from
> the public guidance named in each `source` field. They are teaching material, not
> verbatim reproductions, and are not a substitute for the guidelines themselves.

The next cell writes the corpus to disk so the notebook is self-contained - there is
nothing to upload.

In [8]:
# The corpus, embedded so this notebook runs stand-alone. Written to ./corpus/
# so that `load_corpus()` below is the same code path used outside Colab.
CORPUS_FILES = {
    "01_hypertension.md": "---\nid: hypertension\ntitle: Hypertension - Classification, Diagnosis, and Management\nsource: ACC/AHA 2017 Guideline for High Blood Pressure in Adults; WHO Guideline for the Pharmacological Treatment of Hypertension (2021)\ncategory: cardiovascular\n---\nHypertension is persistently elevated arterial blood pressure and is the leading modifiable risk factor for stroke, myocardial infarction, heart failure, and chronic kidney disease. It is usually asymptomatic, so diagnosis depends on measurement rather than symptoms.\n\nBlood pressure categories (ACC/AHA 2017, adults, mmHg):\n- Normal: systolic below 120 AND diastolic below 80.\n- Elevated: systolic 120-129 AND diastolic below 80.\n- Stage 1 hypertension: systolic 130-139 OR diastolic 80-89.\n- Stage 2 hypertension: systolic 140 or higher OR diastolic 90 or higher.\n- Hypertensive crisis: systolic above 180 and/or diastolic above 120, requiring prompt evaluation.\n\nThe WHO 2021 guideline instead uses a threshold of 140/90 mmHg to define hypertension for the purpose of starting drug treatment, so published thresholds differ between guidelines.\n\nDiagnosis should not rest on a single reading. Confirm with the average of two or more readings on two or more separate occasions, using a validated, correctly sized cuff after the patient has rested for five minutes. Out-of-office measurement with ambulatory (ABPM) or home blood pressure monitoring is recommended to confirm the diagnosis and to exclude white-coat hypertension.\n\nInitial workup screens for target-organ damage and secondary causes: basic metabolic panel including creatinine and electrolytes, fasting glucose or HbA1c, lipid profile, urinalysis with urine albumin-to-creatinine ratio, complete blood count, thyroid-stimulating hormone, and a 12-lead ECG.\n\nManagement begins with lifestyle change for all patients: weight loss, the DASH dietary pattern, sodium reduction to under 2 g per day of sodium (about 5 g per day of salt), increased dietary potassium unless contraindicated by kidney disease, 90-150 minutes per week of aerobic exercise, limiting alcohol, and smoking cessation.\n\nFirst-line drug classes are thiazide or thiazide-like diuretics, ACE inhibitors, angiotensin receptor blockers (ARBs), and dihydropyridine calcium channel blockers. ACE inhibitors and ARBs must never be combined with each other, and both are contraindicated in pregnancy. Many patients need two or more agents, and single-pill combinations improve adherence. A common target is below 130/80 mmHg in patients at higher cardiovascular risk. Beta-blockers are not first-line for uncomplicated hypertension but are indicated when there is a compelling comorbidity such as heart failure with reduced ejection fraction or prior myocardial infarction.\n",
    "02_type2_diabetes.md": "---\nid: type2_diabetes\ntitle: Type 2 Diabetes Mellitus - Diagnosis and Glycaemic Management\nsource: American Diabetes Association Standards of Care in Diabetes; WHO Classification of Diabetes Mellitus\ncategory: endocrine\n---\nType 2 diabetes mellitus is a chronic disorder of hyperglycaemia caused by insulin resistance together with progressive beta-cell dysfunction. It accounts for roughly 90-95% of all diabetes and is strongly associated with overweight, physical inactivity, and family history.\n\nDiagnostic criteria (any one of the following; in the absence of unequivocal hyperglycaemia the abnormal result must be confirmed by repeat testing):\n- Fasting plasma glucose 126 mg/dL (7.0 mmol/L) or higher, after no caloric intake for at least 8 hours.\n- Two-hour plasma glucose 200 mg/dL (11.1 mmol/L) or higher during a 75 g oral glucose tolerance test.\n- HbA1c 6.5% (48 mmol/mol) or higher, using a standardised, NGSP-certified assay.\n- Random plasma glucose 200 mg/dL (11.1 mmol/L) or higher in a patient with classic symptoms of hyperglycaemia or hyperglycaemic crisis.\n\nPrediabetes is defined by fasting plasma glucose 100-125 mg/dL (impaired fasting glucose), two-hour OGTT glucose 140-199 mg/dL (impaired glucose tolerance), or HbA1c 5.7-6.4%. HbA1c is unreliable when red cell turnover is abnormal, for example in sickle cell disease, recent transfusion, pregnancy, haemolysis, or advanced kidney disease; use plasma glucose criteria in these settings.\n\nClassic symptoms include polyuria, polydipsia, unexplained weight loss, blurred vision, fatigue, and recurrent infections, but many patients are asymptomatic at diagnosis.\n\nManagement combines medical nutrition therapy, at least 150 minutes per week of moderate physical activity, weight management, diabetes self-management education, and smoking cessation. Metformin remains a common first-line pharmacological agent for people without contraindications. For patients with established atherosclerotic cardiovascular disease, heart failure, or chronic kidney disease, an SGLT2 inhibitor or a GLP-1 receptor agonist is recommended for organ protection independent of HbA1c. A general glycaemic target is HbA1c below 7% (53 mmol/mol), individualised to be less strict in older adults, those with limited life expectancy, or a history of severe hypoglycaemia.\n\nOngoing care includes annual dilated retinal examination, annual urine albumin-to-creatinine ratio and eGFR, comprehensive foot examination with monofilament testing, blood pressure and lipid management, and vaccination.\n",
    "03_asthma.md": "---\nid: asthma\ntitle: Asthma - Diagnosis and Stepwise Treatment\nsource: Global Initiative for Asthma (GINA) Strategy Report\ncategory: respiratory\n---\nAsthma is a chronic inflammatory airway disease characterised by variable expiratory airflow limitation and a history of respiratory symptoms - wheeze, shortness of breath, chest tightness, and cough - that vary over time and in intensity.\n\nDiagnosis requires both a compatible symptom pattern and objective evidence of variable expiratory airflow limitation. The usual confirmation is spirometry showing a reduced FEV1/FVC ratio with a positive bronchodilator reversibility test: an increase in FEV1 of more than 12% and more than 200 mL in adults, 10-15 minutes after an inhaled short-acting beta-2 agonist. Alternative evidence includes excessive variability in twice-daily peak expiratory flow over two weeks, significant improvement after four weeks of anti-inflammatory treatment, or a positive bronchial challenge test.\n\nCommon triggers are viral respiratory infections, allergens such as house dust mite and pollen, exercise, cold air, tobacco smoke, air pollution, occupational exposures, aspirin or NSAIDs in susceptible patients, and non-selective beta-blockers.\n\nA central GINA recommendation is that asthma in adults and adolescents should not be treated with a short-acting beta-2 agonist (SABA) alone. SABA-only treatment increases the risk of severe exacerbations and death. Every adult and adolescent should receive inhaled corticosteroid (ICS)-containing treatment.\n\nThe preferred approach in mild asthma is as-needed low-dose ICS-formoterol used as the reliever. In more persistent disease, maintenance-and-reliever therapy (MART) uses low-dose ICS-formoterol both as regular maintenance and as the reliever. Treatment is stepped up to medium-dose ICS-formoterol, and in severe disease specialist assessment considers add-on long-acting muscarinic antagonists or biologic therapy targeting type 2 inflammation.\n\nAssessment of control looks at daytime symptoms more than twice a week, any night waking due to asthma, reliever use for symptoms more than twice a week, and activity limitation. Every review should include inhaler technique, adherence, trigger and comorbidity management, a written asthma action plan, and step-down consideration after three months of good control.\n\nAn acute severe exacerbation is treated with repeated inhaled SABA, controlled oxygen targeting a saturation of 93-95% in adults, systemic corticosteroids, and ipratropium in severe cases, with urgent hospital transfer if the response is poor.\n",
    "04_copd.md": "---\nid: copd\ntitle: Chronic Obstructive Pulmonary Disease (COPD)\nsource: Global Initiative for Chronic Obstructive Lung Disease (GOLD) Report\ncategory: respiratory\n---\nChronic obstructive pulmonary disease is a heterogeneous lung condition causing persistent, often progressive airflow obstruction due to abnormalities of the airways (bronchitis, bronchiolitis) and of the alveoli (emphysema). The dominant risk factor globally is tobacco smoking; other important exposures are biomass fuel smoke, occupational dusts and fumes, air pollution, childhood respiratory infection, and alpha-1 antitrypsin deficiency.\n\nDiagnosis requires spirometry. A post-bronchodilator FEV1/FVC ratio below 0.70 confirms persistent airflow limitation. Spirometry must be performed after a bronchodilator, because pre-bronchodilator values overestimate obstruction. Severity of obstruction is graded by FEV1 percent predicted: GOLD 1 mild, 80% or above; GOLD 2 moderate, 50-79%; GOLD 3 severe, 30-49%; GOLD 4 very severe, below 30%.\n\nTypical presentation is chronic and progressive dyspnoea, chronic cough, and sputum production in a patient over 40 with a relevant exposure history. Symptom burden is graded with the modified Medical Research Council (mMRC) dyspnoea scale or the COPD Assessment Test (CAT), and patients are grouped by symptoms and exacerbation history into groups A, B, and E to guide initial therapy.\n\nNon-pharmacological management is central: smoking cessation is the single intervention that most alters the course of the disease; pulmonary rehabilitation improves dyspnoea, exercise capacity, and quality of life; and influenza, pneumococcal, COVID-19, RSV, and pertussis vaccination reduce exacerbations.\n\nInhaled bronchodilators are the foundation of drug therapy. Initial treatment is a long-acting muscarinic antagonist (LAMA), a long-acting beta-2 agonist (LABA), or a LAMA plus LABA combination depending on symptom burden. Inhaled corticosteroids are added mainly for patients with frequent exacerbations, particularly when the blood eosinophil count is elevated; ICS use raises the risk of pneumonia. Unlike in asthma, ICS monotherapy is not appropriate in COPD.\n\nLong-term oxygen therapy is indicated for severe resting hypoxaemia, defined as a PaO2 of 55 mmHg or less, or 60 mmHg or less with cor pulmonale or polycythaemia. An exacerbation is treated with short-acting bronchodilators, a short course of systemic corticosteroids, antibiotics when sputum is purulent or ventilatory support is needed, and non-invasive ventilation for hypercapnic respiratory failure.\n",
    "05_pneumonia.md": "---\nid: pneumonia\ntitle: Community-Acquired Pneumonia in Adults\nsource: ATS/IDSA Clinical Practice Guideline on Community-Acquired Pneumonia; NICE Pneumonia Guideline\ncategory: infectious_disease\n---\nCommunity-acquired pneumonia (CAP) is an acute infection of the lung parenchyma acquired outside hospital. Streptococcus pneumoniae remains the most commonly identified bacterial cause; other pathogens include Haemophilus influenzae, Mycoplasma pneumoniae, Chlamydophila pneumoniae, Legionella pneumophila, Staphylococcus aureus, and respiratory viruses such as influenza, RSV, and SARS-CoV-2.\n\nTypical features are fever, cough with or without sputum, pleuritic chest pain, dyspnoea, and, especially in older adults, confusion or a fall without classic respiratory symptoms. Examination may reveal tachypnoea, crackles, bronchial breathing, and dullness to percussion. Diagnosis is clinical plus a chest radiograph showing a new infiltrate; chest imaging is recommended when pneumonia is suspected.\n\nSeverity assessment guides the site of care. The CURB-65 score assigns one point each for Confusion, blood Urea above 7 mmol/L (about 19 mg/dL), Respiratory rate 30 breaths per minute or more, Blood pressure with systolic below 90 mmHg or diastolic 60 mmHg or less, and age 65 years or older. A score of 0-1 generally indicates suitability for outpatient treatment, 2 suggests hospital admission or close supervision, and 3 or more indicates severe pneumonia with consideration of intensive care. The Pneumonia Severity Index (PSI) is an alternative with more variables. Clinical judgement, oxygen saturation, and social circumstances always modify the score.\n\nEmpirical antibiotic therapy for healthy outpatients without comorbidities is amoxicillin, or doxycycline or a macrolide where local pneumococcal macrolide resistance is low. Outpatients with comorbidities receive amoxicillin-clavulanate or a cephalosporin combined with a macrolide or doxycycline, or a respiratory fluoroquinolone alone. Inpatients with non-severe CAP receive a beta-lactam plus a macrolide, or a respiratory fluoroquinolone. Severe CAP requires a beta-lactam plus either a macrolide or a fluoroquinolone, with added coverage for MRSA or Pseudomonas aeruginosa only when risk factors are present.\n\nTreatment duration is a minimum of five days, continued until the patient is clinically stable and afebrile for 48 hours. Supportive care includes oxygen, fluids, and analgesia. Prevention rests on pneumococcal and influenza vaccination and smoking cessation. Failure to improve by 48-72 hours should prompt reassessment for resistant organisms, empyema, or an alternative diagnosis.\n",
    "06_stroke.md": "---\nid: stroke\ntitle: Acute Ischaemic Stroke - Recognition and Emergency Treatment\nsource: AHA/ASA Guidelines for the Early Management of Patients with Acute Ischemic Stroke; WHO\ncategory: neurology\n---\nStroke is an acute focal neurological deficit caused by a vascular event. About 85% of strokes are ischaemic, caused by arterial occlusion, and about 15% are haemorrhagic. A transient ischaemic attack (TIA) produces the same deficits without acute infarction and carries a high short-term risk of completed stroke.\n\nPublic recognition uses the FAST mnemonic: Facial drooping, Arm weakness, Speech difficulty, Time to call emergency services. Other sudden features include unilateral numbness, visual loss, vertigo with ataxia, and the worst headache of life, which suggests haemorrhage. Deficit severity is quantified with the NIH Stroke Scale (NIHSS).\n\nStroke is a time-critical emergency; the guiding principle is that time is brain, because roughly 1.9 million neurons are lost per minute of untreated large-vessel occlusion. Immediate non-contrast CT of the head, or MRI, is required to distinguish ischaemic from haemorrhagic stroke before any thrombolytic is given. Capillary blood glucose must be checked immediately, since hypoglycaemia is a common stroke mimic.\n\nIntravenous thrombolysis with alteplase is indicated for eligible patients within 4.5 hours of symptom onset or of the time the patient was last known well; tenecteplase is an accepted alternative. Key contraindications include intracranial haemorrhage, recent intracranial or spinal surgery, active internal bleeding, blood pressure above 185/110 mmHg that cannot be safely lowered, and platelet count below 100,000 per microlitre. Endovascular mechanical thrombectomy is indicated for large-vessel occlusion of the anterior circulation, generally within 6 hours, and in selected patients up to 24 hours when advanced perfusion imaging shows salvageable tissue.\n\nPatients not eligible for reperfusion receive aspirin within 24-48 hours. Supportive care includes maintaining normoglycaemia, treating fever, a swallow screen before any oral intake to prevent aspiration, and permissive hypertension in the acute phase unless thrombolysis is planned.\n\nSecondary prevention targets the cause: antiplatelet therapy for atherothrombotic stroke, anticoagulation for atrial fibrillation, high-intensity statin therapy, blood pressure control, diabetes management, smoking cessation, and carotid revascularisation for significant symptomatic stenosis.\n",
    "07_acs_mi.md": "---\nid: acute_coronary_syndrome\ntitle: Acute Coronary Syndrome and ST-Elevation Myocardial Infarction\nsource: ESC Guidelines for Acute Coronary Syndromes; ACC/AHA Guideline for Coronary Artery Disease\ncategory: cardiovascular\n---\nAcute coronary syndrome (ACS) covers unstable angina, non-ST-elevation myocardial infarction (NSTEMI), and ST-elevation myocardial infarction (STEMI). Most cases are caused by rupture or erosion of an atherosclerotic coronary plaque with superimposed thrombus.\n\nTypical presentation is retrosternal chest pain or pressure lasting more than 20 minutes, sometimes radiating to the left arm, neck, or jaw, with sweating, nausea, or dyspnoea. Atypical or silent presentations are common in women, older adults, and people with diabetes, who may present only with breathlessness, fatigue, syncope, or epigastric discomfort.\n\nA 12-lead ECG must be obtained and interpreted within 10 minutes of first medical contact. STEMI is diagnosed by persistent ST-segment elevation in two contiguous leads or a new left bundle branch block; a posterior infarct may show ST depression in V1-V3 and requires posterior leads. In the absence of ST elevation, serial high-sensitivity cardiac troponin measurement distinguishes NSTEMI, in which troponin rises and falls, from unstable angina, in which it does not.\n\nFor STEMI, reperfusion is the priority. Primary percutaneous coronary intervention (PCI) is preferred, with a target of 90 minutes from first medical contact to device in a PCI-capable centre, or 120 minutes when transfer is required. If PCI cannot be delivered within 120 minutes, fibrinolytic therapy should be given within 30 minutes of arrival, followed by transfer for angiography.\n\nInitial medical therapy includes aspirin 150-300 mg chewed, a P2Y12 inhibitor such as ticagrelor or prasugrel (clopidogrel when these are unsuitable), anticoagulation with heparin, and oxygen only if the oxygen saturation is below 90%. Nitrates relieve ischaemic pain but are contraindicated in right ventricular infarction, hypotension, or recent phosphodiesterase-5 inhibitor use.\n\nSecondary prevention after myocardial infarction includes dual antiplatelet therapy for around 12 months, a high-intensity statin, a beta-blocker, an ACE inhibitor or ARB particularly with reduced ejection fraction, and cardiac rehabilitation with structured exercise, smoking cessation, and dietary counselling.\n",
    "08_sepsis.md": "---\nid: sepsis\ntitle: Sepsis and Septic Shock - Recognition and Initial Resuscitation\nsource: Surviving Sepsis Campaign International Guidelines; Sepsis-3 Consensus Definitions\ncategory: critical_care\n---\nSepsis is life-threatening organ dysfunction caused by a dysregulated host response to infection. Under the Sepsis-3 definitions, organ dysfunction is identified as an acute increase of 2 points or more in the Sequential Organ Failure Assessment (SOFA) score. Septic shock is a subset of sepsis in which circulatory and cellular abnormalities are severe enough to substantially increase mortality: it is identified clinically by persisting hypotension requiring vasopressors to maintain a mean arterial pressure of 65 mmHg or higher together with a serum lactate above 2 mmol/L despite adequate fluid resuscitation.\n\nBedside screening outside the intensive care unit may use qSOFA, which assigns one point each for respiratory rate of 22 breaths per minute or more, altered mental status, and systolic blood pressure of 100 mmHg or less. qSOFA has poor sensitivity as a single screening tool and should not be used alone; the guidelines recommend systematic screening with tools such as NEWS or SIRS criteria alongside clinical judgement.\n\nThe hour-1 bundle should be started immediately on recognition:\n- Measure serum lactate, and remeasure if the initial value is above 2 mmol/L.\n- Obtain blood cultures before administering antimicrobials, provided this causes no substantial delay.\n- Administer broad-spectrum antimicrobials; in septic shock they should be given within one hour of recognition.\n- Begin rapid administration of 30 mL/kg of intravenous crystalloid for hypotension or a lactate of 4 mmol/L or more.\n- Start vasopressors during or after fluid resuscitation to maintain a mean arterial pressure of 65 mmHg or higher.\n\nNoradrenaline (norepinephrine) is the first-line vasopressor; vasopressin may be added to reduce the noradrenaline dose. Balanced crystalloids are preferred over 0.9% saline. Source control, such as drainage of an abscess or removal of an infected device, should be achieved as soon as it is practical, ideally within 6-12 hours.\n\nOngoing care includes de-escalation of antimicrobials once cultures and sensitivities return, daily review of the need for continued therapy, lung-protective ventilation if acute respiratory distress syndrome develops, glucose control, and venous thromboembolism prophylaxis.\n",
    "09_anaphylaxis.md": "---\nid: anaphylaxis\ntitle: Anaphylaxis - Emergency Recognition and Treatment\nsource: World Allergy Organization Anaphylaxis Guidance; EAACI Anaphylaxis Guideline; Resuscitation Council\ncategory: emergency_medicine\n---\nAnaphylaxis is a severe, rapidly evolving, potentially fatal systemic hypersensitivity reaction. Onset is usually within minutes to two hours of exposure. The commonest triggers are foods (peanut, tree nuts, shellfish, milk, egg), drugs (beta-lactam antibiotics, NSAIDs, neuromuscular blocking agents, contrast media), and insect venom from bees and wasps.\n\nAnaphylaxis is likely when there is acute onset of illness with involvement of skin or mucosa (urticaria, flushing, angioedema) plus either respiratory compromise (stridor, wheeze, hypoxaemia) or reduced blood pressure or end-organ dysfunction. It is also likely when two or more body systems are rapidly involved after exposure to a probable allergen, or when there is hypotension alone after exposure to a known allergen. Importantly, skin changes are absent in up to 10-20% of cases, so their absence does not exclude anaphylaxis.\n\nIntramuscular adrenaline (epinephrine) is the first-line treatment and must not be delayed; delay is strongly associated with fatal outcome. There is no absolute contraindication to adrenaline in anaphylaxis. The adult dose is 0.3-0.5 mg of the 1 mg/mL (1:1000) solution given intramuscularly into the anterolateral thigh (vastus lateralis). The paediatric dose is 0.01 mg/kg of the same 1 mg/mL solution, to a maximum of 0.3-0.5 mg. The dose may be repeated every 5-15 minutes if the response is inadequate. Intravenous adrenaline is reserved for refractory shock or cardiac arrest and requires an infusion with cardiac monitoring in an appropriate setting.\n\nConcurrent measures are removal of the trigger, calling for emergency help, placing the patient supine with legs elevated (or sitting upright if breathing is the dominant problem, and in the left lateral position if pregnant), high-flow oxygen, and rapid intravenous fluid boluses of crystalloid for hypotension. Sudden change from a supine to an upright posture can precipitate cardiac arrest and should be avoided.\n\nAntihistamines and corticosteroids are second-line adjuncts only. They relieve cutaneous symptoms but do not treat airway obstruction or shock and must never replace or delay adrenaline.\n\nBecause biphasic reactions occur in a minority of patients, observation for 6-12 hours is advised after significant reactions. On discharge, prescribe two adrenaline autoinjectors, train the patient in their use, provide a written emergency plan, and refer to an allergy specialist.\n",
    "10_malaria.md": "---\nid: malaria\ntitle: Malaria - Diagnosis and Treatment\nsource: WHO Guidelines for Malaria; WHO Malaria Fact Sheet\ncategory: infectious_disease\n---\nMalaria is a febrile illness caused by Plasmodium parasites transmitted through the bite of infected female Anopheles mosquitoes. Five species infect humans: P. falciparum, P. vivax, P. ovale, P. malariae, and P. knowlesi. P. falciparum causes most severe disease and deaths, particularly in sub-Saharan Africa, while P. vivax and P. ovale form dormant liver hypnozoites that cause relapse weeks to months later.\n\nThe incubation period is usually 7-30 days. Symptoms are non-specific: fever, chills and rigors, headache, myalgia, fatigue, nausea, vomiting, and diarrhoea. Malaria must be considered in any febrile patient who has been in an endemic area within the previous year.\n\nParasitological confirmation is required before treatment wherever possible. Microscopy of thick and thin blood films remains the reference standard, allowing species identification and parasite density. Rapid diagnostic tests detecting HRP2 or pLDH antigens are widely used; some P. falciparum strains have hrp2/hrp3 gene deletions and can give false-negative HRP2-based results. A single negative film does not exclude malaria, so repeat testing every 12-24 hours for up to three sets if suspicion remains.\n\nFeatures of severe malaria include impaired consciousness or coma, prostration, multiple convulsions, respiratory distress or acidotic breathing, pulmonary oedema, circulatory collapse, abnormal bleeding, jaundice, haemoglobin below 7 g/dL in children, hypoglycaemia below 2.2 mmol/L, acute kidney injury, and a parasitaemia above 10%.\n\nUncomplicated P. falciparum malaria is treated with an artemisinin-based combination therapy (ACT) for three days; examples include artemether-lumefantrine, artesunate-amodiaquine, and dihydroartemisinin-piperaquine. Monotherapy with an artemisinin derivative must be avoided because it promotes resistance. Severe malaria of any species is treated with intravenous artesunate for at least 24 hours, followed by a full course of ACT once the patient can take oral medication; intravenous artesunate is superior to quinine and reduces mortality. For P. vivax and P. ovale, radical cure with primaquine or tafenoquine eliminates hypnozoites, and G6PD status must be checked first because these drugs cause haemolysis in G6PD deficiency.\n\nPrevention relies on insecticide-treated nets, indoor residual spraying, intermittent preventive treatment in pregnancy, seasonal chemoprevention in children, chemoprophylaxis for travellers, and the RTS,S/AS01 and R21/Matrix-M vaccines in endemic settings.\n",
    "11_tuberculosis.md": "---\nid: tuberculosis\ntitle: Tuberculosis - Diagnosis and Treatment Regimens\nsource: WHO Consolidated Guidelines on Tuberculosis; WHO Tuberculosis Fact Sheet\ncategory: infectious_disease\n---\nTuberculosis (TB) is caused by Mycobacterium tuberculosis and spreads through airborne droplet nuclei from a person with pulmonary or laryngeal disease. It most often affects the lungs but can involve lymph nodes, pleura, bone, meninges, and other sites. Infection may remain latent, without symptoms or infectiousness, and progress to active disease later; the lifetime risk of progression is about 5-10% in immunocompetent people and much higher in people living with HIV.\n\nClassic symptoms of pulmonary TB are a cough lasting two weeks or more, fever, drenching night sweats, unintentional weight loss, and haemoptysis. Major risk factors are HIV infection, diabetes, malnutrition, silicosis, smoking, alcohol use disorder, immunosuppressive therapy, and crowded living conditions.\n\nWHO recommends rapid molecular tests as the initial diagnostic test rather than smear microscopy. Nucleic acid amplification tests such as Xpert MTB/RIF Ultra detect M. tuberculosis and rifampicin resistance within hours from a single sputum specimen. Culture remains the reference standard and allows full drug-susceptibility testing but takes weeks. Chest radiography supports diagnosis and is used for screening. Latent TB infection is identified by a tuberculin skin test or an interferon-gamma release assay, after active disease has been excluded.\n\nThe standard regimen for drug-susceptible pulmonary TB in adults is six months of treatment: two months of the intensive phase with isoniazid, rifampicin, pyrazinamide, and ethambutol (2HRZE), followed by four months of the continuation phase with isoniazid and rifampicin (4HR). A four-month regimen of rifapentine, isoniazid, pyrazinamide, and moxifloxacin is an alternative for eligible patients. Fixed-dose combination tablets and treatment adherence support are recommended.\n\nKey adverse effects to monitor are drug-induced hepatitis with isoniazid, rifampicin, and pyrazinamide; peripheral neuropathy from isoniazid, prevented with pyridoxine; optic neuritis from ethambutol; and hyperuricaemia from pyrazinamide. Rifampicin induces hepatic enzymes, reducing the effectiveness of oral contraceptives and many antiretrovirals, and turns urine and tears orange.\n\nMultidrug-resistant TB, defined by resistance to at least isoniazid and rifampicin, requires regimens built from bedaquiline, pretomanid, linezolid, and moxifloxacin, such as the six-month BPaLM regimen. Preventive treatment for latent infection uses isoniazid, or shorter rifamycin-based regimens such as three months of weekly rifapentine plus isoniazid.\n",
    "12_ckd.md": "---\nid: chronic_kidney_disease\ntitle: Chronic Kidney Disease - Staging and Management\nsource: KDIGO Clinical Practice Guideline for the Evaluation and Management of Chronic Kidney Disease\ncategory: nephrology\n---\nChronic kidney disease (CKD) is defined as abnormalities of kidney structure or function present for more than three months with implications for health. It is diagnosed when either the estimated glomerular filtration rate (eGFR) is below 60 mL/min/1.73 m2, or there are markers of kidney damage such as albuminuria, urine sediment abnormalities, electrolyte abnormalities from tubular disorders, histological or imaging abnormalities, or a history of kidney transplantation. The three-month requirement distinguishes CKD from acute kidney injury.\n\nCKD is staged by cause, GFR category, and albuminuria category. GFR categories in mL/min/1.73 m2 are: G1 normal or high, 90 or above; G2 mildly decreased, 60-89; G3a mildly to moderately decreased, 45-59; G3b moderately to severely decreased, 30-44; G4 severely decreased, 15-29; and G5 kidney failure, below 15. Albuminuria categories by urine albumin-to-creatinine ratio (ACR) are: A1 normal to mildly increased, below 30 mg/g (3 mg/mmol); A2 moderately increased, 30-300 mg/g; and A3 severely increased, above 300 mg/g. Note that G1 and G2 qualify as CKD only when a marker of kidney damage such as albuminuria is present.\n\nThe leading causes worldwide are diabetes and hypertension; others include glomerulonephritis, polycystic kidney disease, obstructive uropathy, and recurrent injury from nephrotoxins. CKD is usually asymptomatic until advanced, when fatigue, oedema, nausea, pruritus, and reduced urine output appear.\n\nManagement aims to slow progression and reduce cardiovascular risk. Blood pressure control targets a systolic pressure below 120 mmHg where tolerated. An ACE inhibitor or ARB titrated to the maximum tolerated dose is first-line in CKD with albuminuria; a rise in creatinine of up to about 30% after starting is expected and does not require stopping. SGLT2 inhibitors reduce progression and cardiovascular events in CKD with or without diabetes. A non-steroidal mineralocorticoid receptor antagonist such as finerenone gives additional protection in CKD with type 2 diabetes. Statin therapy is recommended for most adults with CKD.\n\nComplications requiring monitoring are anaemia, mineral and bone disorder with elevated phosphate and parathyroid hormone, metabolic acidosis, and hyperkalaemia. Avoid nephrotoxins including NSAIDs, adjust drug doses for kidney function, and hold metformin and other agents during acute illness. Refer to nephrology for eGFR below 30, ACR above 300 mg/g, rapid progression, or an uncertain cause, and prepare for kidney replacement therapy well before stage G5.\n",
    "13_anaemia.md": "---\nid: iron_deficiency_anaemia\ntitle: Iron Deficiency Anaemia - Diagnosis and Treatment\nsource: WHO Haemoglobin Concentrations for the Diagnosis of Anaemia; BSH Guideline for the Management of Iron Deficiency Anaemia\ncategory: haematology\n---\nAnaemia is a reduction in haemoglobin concentration below the threshold for age, sex, and physiological state. WHO thresholds at sea level are haemoglobin below 130 g/L in non-pregnant men aged 15 and over, below 120 g/L in non-pregnant women aged 15 and over, below 110 g/L in pregnancy, and below 110 g/L in children aged 6-59 months. Living at high altitude and smoking raise the thresholds because they raise baseline haemoglobin.\n\nIron deficiency is the single most common cause of anaemia worldwide. Causes differ by population: inadequate dietary intake and increased demand in growth and pregnancy, blood loss from heavy menstrual bleeding, gastrointestinal bleeding from peptic ulcer, malignancy, or hookworm infestation, and malabsorption in coeliac disease or after gastric surgery.\n\nSymptoms include fatigue, exertional dyspnoea, dizziness, headache, palpitations, pallor of the conjunctivae and palms, brittle nails or koilonychia, angular cheilitis, restless legs, and pica.\n\nThe blood film and indices show a microcytic hypochromic picture with low mean corpuscular volume and low mean corpuscular haemoglobin, and often a raised red cell distribution width. Serum ferritin is the single most useful test: a low ferritin confirms iron deficiency. Ferritin is an acute-phase reactant and can be falsely normal or high during inflammation, infection, liver disease, or malignancy, so interpret it with C-reactive protein; in the presence of inflammation, transferrin saturation below 20% supports iron deficiency. The main differential for a microcytic anaemia is thalassaemia trait and anaemia of chronic disease.\n\nIron deficiency anaemia is a sign, not a final diagnosis. The underlying cause must be sought. Men of any age and postmenopausal women with iron deficiency anaemia require investigation of the gastrointestinal tract with upper endoscopy and colonoscopy unless another cause is clear, and coeliac serology should be checked.\n\nTreatment is oral iron, for example ferrous sulfate 200 mg providing 65 mg elemental iron. Evidence supports once-daily or alternate-day dosing rather than multiple daily doses, because a single dose raises hepcidin and blocks absorption of subsequent doses. Vitamin C or orange juice aids absorption; tea, calcium, antacids, and proton pump inhibitors reduce it. Haemoglobin should rise by about 20 g/L over four weeks; treatment continues for three months after haemoglobin normalises to replenish stores. Intravenous iron is used for intolerance of oral iron, malabsorption, ongoing losses exceeding oral replacement, or when rapid correction is needed. Transfusion is reserved for haemodynamic instability or symptomatic severe anaemia.\n",
    "14_uti.md": "---\nid: urinary_tract_infection\ntitle: Urinary Tract Infection in Adults\nsource: IDSA Guidelines for Uncomplicated Cystitis and Pyelonephritis; NICE Urinary Tract Infection Guidelines\ncategory: infectious_disease\n---\nUrinary tract infection (UTI) is infection anywhere along the urinary tract. Lower UTI (cystitis) involves the bladder, while upper UTI (pyelonephritis) involves the kidney and is a systemic illness. Escherichia coli causes about 75-85% of uncomplicated infections; other organisms include Klebsiella pneumoniae, Proteus mirabilis, Enterococcus species, and Staphylococcus saprophyticus in young women.\n\nAn uncomplicated UTI occurs in a non-pregnant adult with a structurally and functionally normal urinary tract. A complicated UTI is one associated with pregnancy, male sex, urinary obstruction, catheters, stones, immunosuppression, poorly controlled diabetes, recent instrumentation, or renal impairment.\n\nCystitis presents with dysuria, urinary frequency and urgency, suprapubic pain, and sometimes haematuria, without fever. Acute pyelonephritis adds fever, rigors, flank or loin pain, costovertebral angle tenderness, nausea, and vomiting. In older adults, presentation may be non-specific, and confusion alone is not sufficient evidence of UTI.\n\nDiagnosis in a healthy non-pregnant woman with typical symptoms is clinical, and dipstick testing showing nitrites or leucocyte esterase adds support. Urine culture is not needed for uncomplicated cystitis but is recommended in pregnancy, in men, in suspected pyelonephritis, in recurrent or treatment-failing infection, and in complicated cases. Asymptomatic bacteriuria should not be treated except in pregnancy or before urological procedures that breach the mucosa; treating it otherwise drives resistance without benefit.\n\nFirst-line agents for uncomplicated cystitis are nitrofurantoin for 5 days, trimethoprim-sulfamethoxazole for 3 days where local resistance is below about 20%, or fosfomycin as a single 3 g dose. Nitrofurantoin should be avoided when the creatinine clearance is below about 30-45 mL/min and does not treat pyelonephritis because it does not reach therapeutic tissue concentrations in the kidney. Fluoroquinolones are reserved because of tendon, neurological, and aortic adverse effects.\n\nAcute pyelonephritis is treated for 7-14 days with an oral fluoroquinolone or an intravenous agent such as ceftriaxone when the patient is systemically unwell, vomiting, pregnant, or has failed oral therapy. Recurrent UTI in women is managed with increased fluid intake, review of contraception, vaginal oestrogen after the menopause, and, in selected cases, prophylaxis. Imaging is indicated for suspected obstruction, stones, or failure to improve within 48-72 hours.\n",
    "15_hypothyroidism.md": "---\nid: hypothyroidism\ntitle: Hypothyroidism - Diagnosis and Levothyroxine Therapy\nsource: American Thyroid Association Guidelines for the Treatment of Hypothyroidism; ETA Guideline on Subclinical Hypothyroidism\ncategory: endocrine\n---\nHypothyroidism is a deficiency of thyroid hormone. Primary hypothyroidism, caused by disease of the thyroid gland itself, accounts for the great majority of cases. Worldwide the commonest cause is iodine deficiency; in iodine-sufficient regions it is chronic autoimmune (Hashimoto) thyroiditis. Other causes are thyroidectomy, radioactive iodine therapy, external neck irradiation, and drugs such as amiodarone, lithium, interferon-alfa, and immune checkpoint inhibitors. Central hypothyroidism from pituitary or hypothalamic disease is rare.\n\nSymptoms develop insidiously and are non-specific: fatigue, cold intolerance, weight gain, constipation, dry skin, hair thinning, hoarseness, muscle cramps, menstrual irregularity, low mood, and slowed cognition. Signs may include bradycardia, a delayed relaxation phase of the deep tendon reflexes, periorbital puffiness, and goitre.\n\nSerum thyroid-stimulating hormone (TSH) is the first-line test because the log-linear relationship between TSH and free T4 makes TSH highly sensitive to small changes in thyroid hormone. In overt primary hypothyroidism TSH is elevated and free T4 is low. In subclinical hypothyroidism TSH is elevated while free T4 remains within the reference range. A low or inappropriately normal TSH with a low free T4 suggests central hypothyroidism and requires pituitary assessment rather than reliance on TSH. Thyroid peroxidase (TPO) antibodies confirm autoimmune aetiology and predict progression from subclinical to overt disease. TSH should not be used to diagnose thyroid dysfunction during acute non-thyroidal illness because of transient abnormalities.\n\nTreatment of overt hypothyroidism is oral levothyroxine (LT4) monotherapy, which is the standard of care. A typical full replacement dose is approximately 1.6 micrograms per kilogram of body weight per day. Older adults and those with coronary artery disease should start at a low dose, such as 12.5-25 micrograms daily, and titrate slowly to avoid precipitating angina or arrhythmia. Levothyroxine is best taken on an empty stomach, 30-60 minutes before breakfast or at bedtime several hours after eating; calcium, iron, proton pump inhibitors, and soy reduce absorption and should be separated by at least four hours.\n\nRecheck TSH 6-8 weeks after starting or changing a dose, since the pituitary takes that long to re-equilibrate, and thereafter every 6-12 months once stable. The usual goal is a TSH within the reference range. In pregnancy, levothyroxine requirements rise by 25-50% early in the first trimester and TSH must be monitored closely, because untreated maternal hypothyroidism harms fetal neurodevelopment. Routine combination therapy with liothyronine (T3) is not recommended.\n",
    "16_amr_stewardship.md": "---\nid: antimicrobial_resistance\ntitle: Antimicrobial Resistance and Antibiotic Stewardship\nsource: WHO Antimicrobial Resistance Fact Sheet; WHO AWaRe Classification of Antibiotics; CDC Core Elements of Antibiotic Stewardship\ncategory: public_health\n---\nAntimicrobial resistance (AMR) occurs when bacteria, viruses, fungi, and parasites no longer respond to the medicines used to treat them, making infections harder to treat and increasing the risk of severe illness, spread, and death. AMR is one of the leading global public health threats; bacterial AMR is associated with millions of deaths each year and directly causes a large share of them. It arises through natural selection but is accelerated by misuse and overuse of antimicrobials in humans, animals, and agriculture, by poor infection prevention, and by inadequate access to clean water and sanitation.\n\nPriority resistant organisms include carbapenem-resistant Enterobacterales, extended-spectrum beta-lactamase (ESBL)-producing Enterobacterales, carbapenem-resistant Acinetobacter baumannii and Pseudomonas aeruginosa, methicillin-resistant Staphylococcus aureus (MRSA), vancomycin-resistant Enterococcus, drug-resistant Mycobacterium tuberculosis, and drug-resistant Neisseria gonorrhoeae.\n\nThe WHO AWaRe classification groups antibiotics into three categories to guide use. Access antibiotics are first- and second-choice agents for common infections with a narrow spectrum and lower resistance potential, such as amoxicillin, nitrofurantoin, and doxycycline; WHO sets a target that at least 70% of national antibiotic consumption comes from this group. Watch antibiotics have higher resistance potential and should be used selectively, for example ciprofloxacin, ceftriaxone, and azithromycin. Reserve antibiotics, such as colistin, linezolid, and ceftazidime-avibactam, are last-resort agents for confirmed or suspected multidrug-resistant infections.\n\nAntibiotic stewardship is the coordinated effort to improve appropriate use. Core practices are: prescribe antibiotics only when a bacterial infection is likely, since most acute respiratory infections including the common cold, most sore throats, and acute bronchitis are viral and do not benefit; take appropriate cultures before starting therapy; choose the narrowest effective agent using local antibiograms; document the indication, dose, and planned duration; review at 48-72 hours to de-escalate, switch from intravenous to oral, or stop; and use the shortest effective course, as shorter courses are now supported for many infections.\n\nComplementary measures are hand hygiene, vaccination, infection prevention and control, safe water and sanitation, surveillance of resistance and consumption, restriction of antibiotics for growth promotion in animals, and patient education that antibiotics do not treat viral illness and that courses should not be shared or saved.\n"
}

Path(CORPUS_DIR).mkdir(exist_ok=True)
for _filename, _content in CORPUS_FILES.items():
    (Path(CORPUS_DIR) / _filename).write_text(_content, encoding='utf-8')

print(f'Wrote {len(CORPUS_FILES)} documents to ./{CORPUS_DIR}/')

Wrote 16 documents to ./corpus/


In [9]:
@dataclass
class Document:
    """One trusted medical reference document plus its metadata."""

    id: str
    title: str
    source: str
    category: str
    text: str


def clean_text(raw: str) -> str:
    """Light-touch cleaning - only what this corpus actually needs.

    The corpus is hand-written Markdown, so there is no messy scraped HTML to
    fight. We still guard against the three things that would hurt retrieval:
    stray HTML tags, runaway blank lines, and repeated spaces (which change how
    the text is tokenised, and therefore its embedding).
    """
    text = re.sub(r"<[^>]+>", " ", raw)      # drop any HTML tags
    text = re.sub(r"[ \t]+", " ", text)      # collapse runs of spaces and tabs
    text = re.sub(r"\n{3,}", "\n\n", text)   # at most one blank line between paragraphs
    lines = [line.strip() for line in text.splitlines()]
    return "\n".join(lines).strip()


def parse_document(raw: str, filename: str = "") -> Document:
    """Read one corpus file: a `---` metadata block followed by the body text."""
    meta: dict[str, str] = {}
    body = raw
    if raw.lstrip().startswith("---"):
        _, front_matter, body = raw.lstrip().split("---", 2)
        for line in front_matter.strip().splitlines():
            if ":" in line:
                key, value = line.split(":", 1)
                meta[key.strip()] = value.strip()
    stem = Path(filename).stem
    return Document(
        id=meta.get("id", stem),
        title=meta.get("title", stem),
        source=meta.get("source", "unknown"),
        category=meta.get("category", "general"),
        text=clean_text(body),
    )


def load_corpus(corpus_dir: str = CORPUS_DIR) -> list[Document]:
    """Load every Markdown document in the corpus directory, sorted by filename."""
    files = sorted(Path(corpus_dir).glob("*.md"))
    if not files:
        raise FileNotFoundError(f"No .md documents found in {Path(corpus_dir).resolve()}")
    return [parse_document(f.read_text(encoding="utf-8"), f.name) for f in files]

In [10]:
documents = load_corpus()
print(f"Loaded {len(documents)} documents\n")

for doc in documents:
    print(f"{doc.id:28s} {doc.category:18s} {len(doc.text.split()):4d} words  {doc.title}")

# One document in full, so the metadata and the cleaned text are both visible.
print("\n" + "=" * 78)
example = documents[0]
print(f"id:       {example.id}\ntitle:    {example.title}\n"
      f"source:   {example.source}\ncategory: {example.category}\n")
print(example.text[:700] + " ...")

Loaded 16 documents

hypertension                 cardiovascular      356 words  Hypertension - Classification, Diagnosis, and Management
type2_diabetes               endocrine           322 words  Type 2 Diabetes Mellitus - Diagnosis and Glycaemic Management
asthma                       respiratory         338 words  Asthma - Diagnosis and Stepwise Treatment
copd                         respiratory         332 words  Chronic Obstructive Pulmonary Disease (COPD)
pneumonia                    infectious_disease  338 words  Community-Acquired Pneumonia in Adults
stroke                       neurology           316 words  Acute Ischaemic Stroke - Recognition and Emergency Treatment
acute_coronary_syndrome      cardiovascular      306 words  Acute Coronary Syndrome and ST-Elevation Myocardial Infarction
sepsis                       critical_care       331 words  Sepsis and Septic Shock - Recognition and Initial Resuscitation
anaphylaxis                  emergency_medicine  360 words  Anaphy

---
## Section 5 - Document Processing and Chunking

### Cleaning

The corpus is hand-written Markdown, so there is no scraped HTML or navigation junk to
strip. Cleaning is therefore deliberately small - it only fixes the things that would
actually change an embedding: stray HTML tags, repeated spaces, and runs of blank
lines. Building an elaborate preprocessing pipeline for text that is already clean
would be effort spent on nothing.

### Chunking

Why chunk at all? Two reasons. A whole document averaged into one vector blurs its
distinct topics together, so a question about one detail matches poorly. And feeding
three entire documents to the model wastes context on paragraphs nobody asked about.

**The strategy: pack whole paragraphs into chunks of at most 180 words, repeating the
last paragraph at the start of the next chunk.**

- **Why paragraphs, not a fixed word count?** In this corpus each paragraph is one
  self-contained idea - a symptom list, a dosing rule, a diagnostic threshold. Cutting
  strictly every N words would slice bullet lists in half and leave "Stage 1
  hypertension: systolic 130-139 OR" stranded at a boundary.
- **Why ~180 words?** Documents are 330-425 words, so this yields 3-4 chunks each -
  focused enough to match one topic, complete enough to contain a whole answer. Three
  retrieved chunks make a prompt of roughly 1,000 tokens, comfortable for a small model.
- **Why repeat one paragraph?** Overlap is cheap insurance: a fact that lands right on
  a boundary stays readable in the following chunk too.

Each chunk keeps its parent document's `title` and `source`, which is what lets every
retrieved passage be attributed later.

In [11]:
@dataclass
class Chunk:
    """A retrievable passage.

    It carries its parent document's metadata, so every retrieved result can be
    attributed back to a named medical source.
    """

    chunk_id: int
    doc_id: str
    title: str
    source: str
    text: str


def split_into_chunks(
    text: str,
    max_words: int = MAX_CHUNK_WORDS,
    overlap_paragraphs: int = OVERLAP_PARAGRAPHS,
) -> list[str]:
    """Pack whole paragraphs into chunks of at most `max_words` words.

    Why paragraphs: this corpus keeps one idea per paragraph (a symptom list, a
    dosing rule, a diagnostic threshold), so a paragraph boundary is a natural
    place to cut. Splitting at a fixed word count would slice bullet lists in half.

    Why ~180 words: the documents are 330-425 words each, so this gives 2-3 chunks
    per document - small enough that a chunk is about one topic, large enough to
    contain a complete answer. Three such chunks make a prompt of roughly 500
    words, which fits comfortably in a small model's context window.

    Why repeat the last paragraph: a fact that lands right on a boundary stays
    readable in the following chunk instead of being cut off mid-thought.
    """
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks: list[str] = []
    current: list[str] = []
    current_words = 0

    for paragraph in paragraphs:
        words = len(paragraph.split())
        if current and current_words + words > max_words:
            chunks.append("\n\n".join(current))
            current = current[-overlap_paragraphs:] if overlap_paragraphs else []
            current_words = sum(len(p.split()) for p in current)
        current.append(paragraph)
        current_words += words

    if current:
        chunks.append("\n\n".join(current))
    return chunks


def build_chunks(documents: list[Document]) -> list[Chunk]:
    """Turn the document list into a flat, numbered list of retrievable chunks."""
    chunks: list[Chunk] = []
    for doc in documents:
        for text in split_into_chunks(doc.text):
            chunks.append(
                Chunk(
                    chunk_id=len(chunks),
                    doc_id=doc.id,
                    title=doc.title,
                    source=doc.source,
                    text=text,
                )
            )
    return chunks

In [12]:
chunks = build_chunks(documents)
words = [len(c.text.split()) for c in chunks]

print(f"{len(documents)} documents -> {len(chunks)} chunks")
print(f"words per chunk: min={min(words)}  mean={sum(words)/len(words):.0f}  max={max(words)}")

print("\n" + "=" * 78)
print(f"Example - chunk {chunks[0].chunk_id}, from '{chunks[0].title}':\n")
print(chunks[0].text)

16 documents -> 55 chunks
words per chunk: min=101  mean=148  max=204

Example - chunk 0, from 'Hypertension - Classification, Diagnosis, and Management':

Hypertension is persistently elevated arterial blood pressure and is the leading modifiable risk factor for stroke, myocardial infarction, heart failure, and chronic kidney disease. It is usually asymptomatic, so diagnosis depends on measurement rather than symptoms.

Blood pressure categories (ACC/AHA 2017, adults, mmHg):
- Normal: systolic below 120 AND diastolic below 80.
- Elevated: systolic 120-129 AND diastolic below 80.
- Stage 1 hypertension: systolic 130-139 OR diastolic 80-89.
- Stage 2 hypertension: systolic 140 or higher OR diastolic 90 or higher.
- Hypertensive crisis: systolic above 180 and/or diastolic above 120, requiring prompt evaluation.

The WHO 2021 guideline instead uses a threshold of 140/90 mmHg to define hypertension for the purpose of starting drug treatment, so published thresholds differ between guideline

---
## Section 6 - Generate Embeddings

### What is an embedding?

An embedding is a list of numbers - here 384 of them - that represents the *meaning*
of a piece of text. The model is trained so that texts meaning similar things end up
close together in that 384-dimensional space, even when they share no words.

That is the whole point. Keyword search for *"high blood pressure"* misses a document
that only ever says *"hypertension"*. Their embeddings sit close together, so vector
search finds it. This matters in medicine, where the everyday phrasing a user types
and the clinical term in a guideline are often entirely different words.

We measure closeness with **cosine similarity**: 1.0 means identical direction,
0.0 means unrelated. Because we normalise every vector to unit length, cosine
similarity is just the dot product - which is exactly what FAISS computes in Section 7.

### Why `BAAI/bge-small-en-v1.5`?

- **Open source** (MIT), downloaded once and run locally - no API, no key, no cost.
- **Small:** 33M parameters, ~130 MB. Embedding 55 chunks takes seconds on a free CPU.
- **Strong for its size:** among the best small English retrieval models on the MTEB
  benchmark, and specifically trained for retrieval rather than general similarity.
- **384 dimensions:** compact enough to keep search instant, rich enough to separate
  closely related clinical topics.

**One detail that matters:** BGE models are trained *asymmetrically*. Questions get a
short instruction prefix, passages get none. Skipping that prefix measurably degrades
retrieval, so `embed_query()` adds it and `embed_passages()` does not.

In [13]:
def load_embedding_model(model_name: str = EMBED_MODEL) -> SentenceTransformer:
    """Load the open-source embedding model (downloaded once, then cached)."""
    return SentenceTransformer(model_name)


def embed_passages(model: SentenceTransformer, texts: list[str]) -> np.ndarray:
    """Embed the corpus passages.

    Vectors are L2-normalised, so the dot product between any two of them is
    exactly their cosine similarity. FAISS then needs only an inner-product index.
    """
    return model.encode(
        texts,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype("float32")


def embed_query(model: SentenceTransformer, query: str) -> np.ndarray:
    """Embed one question, with the query prefix that BGE models are trained with."""
    vector = model.encode(
        [QUERY_PREFIX + query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return vector.astype("float32")

In [14]:
embedder = load_embedding_model()
chunk_embeddings = embed_passages(embedder, [c.text for c in chunks])

print(f"\nembedding matrix: {chunk_embeddings.shape}  "
      f"({chunk_embeddings.shape[0]} chunks x {chunk_embeddings.shape[1]} dimensions)")
print(f"dtype: {chunk_embeddings.dtype}")
print(f"L2 norm of first vector: {float((chunk_embeddings[0] ** 2).sum() ** 0.5):.4f}  "
      "(1.0 confirms normalisation -> dot product == cosine similarity)")
print(f"\nfirst 8 values of chunk 0: {chunk_embeddings[0][:8].round(4)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


embedding matrix: (55, 384)  (55 chunks x 384 dimensions)
dtype: float32
L2 norm of first vector: 1.0000  (1.0 confirms normalisation -> dot product == cosine similarity)

first 8 values of chunk 0: [-0.0361  0.0241 -0.0336  0.0004 -0.0168 -0.0148  0.0531  0.0135]


---
## Section 7 - Build the FAISS Index

FAISS is Meta's open-source library for storing vectors and searching them by
similarity. It is the vector database for this project.

**Which index? `IndexFlatIP` - flat, exact, inner product.**

"Flat" means no approximation: the query is compared against all 55 stored vectors and
the true best matches come back every time. That is one 55x384 matrix multiply,
microseconds of work.

The alternatives - IVF, HNSW - exist to avoid scanning millions of vectors. They buy
speed by *approximating*, at the cost of tuning parameters (`nlist`, `nprobe`, `efSearch`)
and some genuine recall loss. At 55 vectors there is no speed left to buy, so they
would add configuration and lose accuracy for nothing. Exact search is both simpler and
strictly better here.

"IP" is inner product. Since Section 6 normalised every vector to unit length, the
inner product *is* cosine similarity - so we get cosine ranking without a separate
normalisation step at query time.

In [15]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """Build an exact inner-product index over the normalised embeddings.

    `IndexFlatIP` compares the query against every stored vector - brute force,
    no approximation. With ~50 chunks of 384 dimensions a search is one small
    matrix multiply, far under a millisecond. An approximate index (IVF, HNSW)
    would add tuning parameters and lose recall while buying no measurable speed
    at this size. Because the vectors are normalised, inner product == cosine.
    """
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index

In [16]:
index = build_faiss_index(chunk_embeddings)

print(f"index type:  {type(index).__name__}")
print(f"vectors:     {index.ntotal}")
print(f"dimensions:  {index.d}")
print(f"trained:     {index.is_trained}  (a flat index needs no training step)")

index type:  IndexFlatIP
vectors:     55
dimensions:  384
trained:     True  (a flat index needs no training step)


---
## Section 8 - Retrieval Function

The retrieval half of RAG, in three steps and deliberately nothing more:

1. Embed the question (with the BGE query prefix).
2. Search the FAISS index.
3. Return the **top 3** chunks, each with its cosine score and source metadata.

`top_k = 3` is the task's requirement, and it is a reasonable default: enough context
that a single mediocre match does not sink the answer, few enough that the prompt stays
short and on-topic.

**What is deliberately absent.** No BM25, no hybrid fusion, no reciprocal rank fusion,
no reranker, no similarity cut-off. Dense search alone answers these questions
correctly (measured in Section 12), so each of those would add moving parts and
explanation burden for no gain. The rule applied throughout: add a component when the
task requires it or the evidence demands it, not because it sounds advanced.

Note that we retrieve the top 3 *chunks*. Because every chunk carries its parent
document's title and source, each result is still fully attributable to a named
document - which is what the requirement is really after.

In [17]:
@dataclass
class Result:
    """One retrieved chunk with its cosine similarity to the question."""

    rank: int
    score: float
    chunk: Chunk


@dataclass
class Retriever:
    """The three things retrieval needs: the embedding model, the FAISS index,
    and the chunks that the index rows point back to."""

    embedder: SentenceTransformer
    index: faiss.IndexFlatIP
    chunks: list[Chunk]

    def retrieve_documents(self, query: str, top_k: int = TOP_K) -> list[Result]:
        """Embed the question, search FAISS, return the top-k most similar chunks."""
        query_vector = embed_query(self.embedder, query)
        scores, indices = self.index.search(query_vector, top_k)
        return [
            Result(rank=rank, score=float(score), chunk=self.chunks[int(idx)])
            for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1)
        ]


def build_retriever(chunks: list[Chunk], embedder: SentenceTransformer | None = None) -> Retriever:
    """Embed every chunk, index it, and return a ready-to-query retriever."""
    embedder = embedder or load_embedding_model()
    embeddings = embed_passages(embedder, [c.text for c in chunks])
    return Retriever(embedder=embedder, index=build_faiss_index(embeddings), chunks=chunks)


def show_retrieval(retriever: Retriever, question: str, top_k: int = TOP_K) -> None:
    """Print a question and its top-k retrieved chunks (retrieval only, no LLM)."""
    print(f"Q: {question}\n")
    for r in retriever.retrieve_documents(question, top_k=top_k):
        preview = " ".join(r.chunk.text.split())[:180]
        print(f"  [{r.rank}] cosine={r.score:.4f}  {r.chunk.title}")
        print(f"      source: {r.chunk.source}")
        print(f"      {preview}...\n")

In [18]:
retriever = Retriever(embedder=embedder, index=index, chunks=chunks)

# Retrieval on its own, before any LLM is involved.
show_retrieval(retriever, "What are the common symptoms of malaria and how is it diagnosed?")

Q: What are the common symptoms of malaria and how is it diagnosed?

  [1] cosine=0.7759  Malaria - Diagnosis and Treatment
      source: WHO Guidelines for Malaria; WHO Malaria Fact Sheet
      Malaria is a febrile illness caused by Plasmodium parasites transmitted through the bite of infected female Anopheles mosquitoes. Five species infect humans: P. falciparum, P. viva...

  [2] cosine=0.7643  Malaria - Diagnosis and Treatment
      source: WHO Guidelines for Malaria; WHO Malaria Fact Sheet
      Parasitological confirmation is required before treatment wherever possible. Microscopy of thick and thin blood films remains the reference standard, allowing species identificatio...

  [3] cosine=0.7027  Malaria - Diagnosis and Treatment
      source: WHO Guidelines for Malaria; WHO Malaria Fact Sheet
      Features of severe malaria include impaired consciousness or coma, prostration, multiple convulsions, respiratory distress or acidotic breathing, pulmonary oedema, circulatory coll...

In [19]:
# A question the corpus does not cover. Note the much lower cosine scores - the
# retriever always returns its 3 nearest neighbours, relevant or not. Recognising
# that the context does not answer the question is the LLM's job, handled by the
# prompt in Section 10.
show_retrieval(retriever, "What is the recommended surgical technique for repairing a torn anterior cruciate ligament?")

Q: What is the recommended surgical technique for repairing a torn anterior cruciate ligament?

  [1] cosine=0.5585  Sepsis and Septic Shock - Recognition and Initial Resuscitation
      source: Surviving Sepsis Campaign International Guidelines; Sepsis-3 Consensus Definitions
      The hour-1 bundle should be started immediately on recognition: - Measure serum lactate, and remeasure if the initial value is above 2 mmol/L. - Obtain blood cultures before admini...

  [2] cosine=0.5376  Anaphylaxis - Emergency Recognition and Treatment
      source: World Allergy Organization Anaphylaxis Guidance; EAACI Anaphylaxis Guideline; Resuscitation Council
      Concurrent measures are removal of the trigger, calling for emergency help, placing the patient supine with legs elevated (or sitting upright if breathing is the dominant problem, ...

  [3] cosine=0.5347  Acute Ischaemic Stroke - Recognition and Emergency Treatment
      source: AHA/ASA Guidelines for the Early Management of Patients wit

---
## Section 9 - Load the Language Model

**`Qwen/Qwen2.5-1.5B-Instruct`** - chosen because it is:

- **Open weights, Apache-2.0** - no API key, no per-token cost, runs entirely in this
  notebook.
- **Small enough to be practical:** 1.5B parameters is ~3 GB in float16, so it loads on
  a free Colab T4 and still runs (slowly) on CPU.
- **Instruction-tuned:** it reliably follows the "use only this context" rule, which a
  base completion model of this size would not.
- **Long enough context:** 32k tokens, far more than the ~1,000-token prompts we build.

The model is only ever asked to *read the retrieved passages and answer from them*.
It is not the source of medical knowledge - the corpus is.

> This cell downloads ~3 GB and is the slowest step. On CPU, expect a minute or two per
> answer in Section 12; a T4 GPU runtime makes it a few seconds.

In [20]:
def load_llm(model_name: str = LLM_MODEL):
    """Load the open-source instruction model. float16 on GPU, float32 on CPU."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    model.eval()
    return tokenizer, model

In [21]:
llm = load_llm()
tokenizer, model = llm
print(f"loaded:     {LLM_MODEL}")
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"device:     {model.device}   dtype: {model.dtype}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded:     Qwen/Qwen2.5-1.5B-Instruct
parameters: 1.54B
device:     cuda:0   dtype: torch.float16


---
## Section 10 - The RAG Pipeline

Now the two halves are joined:

```
question -> embed -> FAISS -> top 3 chunks -> context -> prompt -> LLM -> answer
```

### The grounded prompt

The prompt is where "answer only from the context" is actually enforced. It is kept
simple and does four things:

1. Restricts the model to the supplied context.
2. Gives it an exact sentence to use when the context does not contain the answer -
   an explicit escape hatch, so refusing is easier than inventing.
3. Forbids outside knowledge.
4. Names the things that must never be invented: facts, numbers, dosages, diagnoses,
   treatment recommendations.

The rules go in the *system* turn and the context and question in the *user* turn,
which is the message format Qwen2.5-Instruct was tuned on. Each passage is numbered and
labelled with its source title, so the model is reading attributed material.

Decoding is **greedy** (`do_sample=False`): no randomness, so the same question gives
the same answer every run. For medical text, reproducibility beats variety.

Note how cleanly the two halves separate: `retrieve_documents()` never sees the model,
`generate_answer()` never sees the index, and `answer_question()` just wires them
together.

In [22]:
SYSTEM_PROMPT = """You are a medical information assistant for an educational RAG demonstration.

Answer the user's question using ONLY the provided context.

If the answer cannot be found in the context, say:
"I could not find sufficient information in the provided medical knowledge base."

Do not use outside knowledge.
Do not invent facts, numbers, dosages, diagnoses, or treatment recommendations."""

USER_PROMPT = """Context:
{context}

Question:
{question}

Answer:"""


def format_context(results: list[Result]) -> str:
    """Render the retrieved chunks as numbered passages for the prompt."""
    return "\n\n".join(
        f"[{r.rank}] {r.chunk.title} (Source: {r.chunk.source})\n{r.chunk.text}"
        for r in results
    )


def build_prompt(question: str, context: str) -> list[dict]:
    """The grounded prompt: rules in the system turn, context + question in the user turn."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT.format(context=context, question=question)},
    ]


def generate_answer(tokenizer, model, question: str, context: str) -> str:
    """Send the grounded prompt to the LLM and return only the newly generated text."""
    messages = build_prompt(question, context)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,  # greedy decoding -> the same answer on every run
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def answer_question(question: str, retriever: Retriever, llm, top_k: int = TOP_K) -> dict:
    """The complete pipeline: question -> embedding -> FAISS -> top-k -> context -> LLM."""
    results = retriever.retrieve_documents(question, top_k=top_k)
    context = format_context(results)
    tokenizer, model = llm
    answer = generate_answer(tokenizer, model, question, context)
    return {
        "question": question,
        "answer": answer,
        "retrieved": [
            {"rank": r.rank, "score": round(r.score, 4), "doc_id": r.chunk.doc_id,
             "title": r.chunk.title, "source": r.chunk.source}
            for r in results
        ],
    }

In [23]:
# Inspect the exact prompt the model receives - nothing hidden.
_q = "What blood pressure readings define stage 2 hypertension in adults?"
_context = format_context(retriever.retrieve_documents(_q))
_prompt = tokenizer.apply_chat_template(build_prompt(_q, _context),
                                        tokenize=False, add_generation_prompt=True)

print(f"prompt length: {len(tokenizer(_prompt)['input_ids'])} tokens\n")
print(_prompt[:1200] + "\n\n   ...context continues...\n")
print(_prompt[-220:])

prompt length: 980 tokens

<|im_start|>system
You are a medical information assistant for an educational RAG demonstration.

Answer the user's question using ONLY the provided context.

If the answer cannot be found in the context, say:
"I could not find sufficient information in the provided medical knowledge base."

Do not use outside knowledge.
Do not invent facts, numbers, dosages, diagnoses, or treatment recommendations.<|im_end|>
<|im_start|>user
Context:
[1] Hypertension - Classification, Diagnosis, and Management (Source: ACC/AHA 2017 Guideline for High Blood Pressure in Adults; WHO Guideline for the Pharmacological Treatment of Hypertension (2021))
Hypertension is persistently elevated arterial blood pressure and is the leading modifiable risk factor for stroke, myocardial infarction, heart failure, and chronic kidney disease. It is usually asymptomatic, so diagnosis depends on measurement rather than symptoms.

Blood pressure categories (ACC/AHA 2017, adults, mmHg):
- Normal:

---
## Section 11 - Sample Questions

Seven questions: **six covered by the corpus** and **one deliberately outside it**.

The in-corpus questions are specific rather than vague - they ask for thresholds,
doses, scores and regimens. That is on purpose: a question with a precise answer makes
it obvious whether the response actually came from the retrieved text or was invented.

The `expected` field names the document a clinician would expect to be cited. It is the
ground truth for the retrieval evaluation in Section 12. The out-of-scope question is
labelled `None` - it tests whether the system admits the gap instead of guessing.

In [24]:
# `expected` names the document a clinician would expect to be cited. It is the
# ground truth for the retrieval evaluation below. `None` marks a question that
# is deliberately outside the knowledge base.
SAMPLE_QUESTIONS = [
    {"question": "What blood pressure readings define stage 2 hypertension in adults?",
     "expected": "hypertension"},
    {"question": "Which HbA1c value confirms a diagnosis of diabetes, and what range counts as prediabetes?",
     "expected": "type2_diabetes"},
    {"question": "What are the common symptoms of malaria and how is it diagnosed?",
     "expected": "malaria"},
    {"question": "What is the adult dose of adrenaline in anaphylaxis and where should it be injected?",
     "expected": "anaphylaxis"},
    {"question": "How is the CURB-65 score calculated in community-acquired pneumonia?",
     "expected": "pneumonia"},
    {"question": "What is the standard six-month drug regimen for drug-susceptible pulmonary tuberculosis?",
     "expected": "tuberculosis"},
    # Deliberately outside the corpus: nothing here covers orthopaedic surgery.
    {"question": "What is the recommended surgical technique for repairing a torn anterior cruciate ligament?",
     "expected": None},
]

In [25]:
for i, item in enumerate(SAMPLE_QUESTIONS, start=1):
    scope = item["expected"] or "OUT OF SCOPE"
    print(f"{i}. [{scope}]\n   {item['question']}\n")

1. [hypertension]
   What blood pressure readings define stage 2 hypertension in adults?

2. [type2_diabetes]
   Which HbA1c value confirms a diagnosis of diabetes, and what range counts as prediabetes?

3. [malaria]
   What are the common symptoms of malaria and how is it diagnosed?

4. [anaphylaxis]
   What is the adult dose of adrenaline in anaphylaxis and where should it be injected?

5. [pneumonia]
   How is the CURB-65 score calculated in community-acquired pneumonia?

6. [tuberculosis]
   What is the standard six-month drug regimen for drug-susceptible pulmonary tuberculosis?

7. [OUT OF SCOPE]
   What is the recommended surgical technique for repairing a torn anterior cruciate ligament?



---
## Section 12 - Retrieval and Answer Demonstration

Each question below is run through the full pipeline, printing **the question**, **the
top 3 retrieved documents with their cosine scores**, and **the generated answer** - so
it is visible that the answer came from the retrieved passages and not from the model's
memory.

Watch the last question in particular: the retrieved passages are unrelated (low cosine
scores) and the model should say it cannot find the information, rather than answering
from what it happens to know about knee surgery.

In [26]:
def show_answer(result: dict) -> None:
    """Print one full RAG result: question, retrieved titles, generated answer."""
    print("=" * 78)
    print(f"QUESTION: {result['question']}\n")
    print("RETRIEVED (top 3):")
    for r in result["retrieved"]:
        print(f"  [{r['rank']}] cosine={r['score']:.4f}  {r['title']}")
    print(f"\nANSWER:\n{result['answer']}\n")


def run_demo(retriever: Retriever, llm, questions: list[dict] = None,
             out_path: str | None = None) -> list[dict]:
    """Answer every sample question and optionally save the results as JSON."""
    questions = questions or SAMPLE_QUESTIONS
    results = []
    for item in questions:
        result = answer_question(item["question"], retriever, llm)
        show_answer(result)
        results.append(result)
    if out_path:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        Path(out_path).write_text(json.dumps(results, indent=2), encoding="utf-8")
        print(f"Saved {len(results)} question/answer pairs to {out_path}")
    return results

In [27]:
results = run_demo(retriever, llm, out_path="outputs/sample_qa.json")

QUESTION: What blood pressure readings define stage 2 hypertension in adults?

RETRIEVED (top 3):
  [1] cosine=0.8187  Hypertension - Classification, Diagnosis, and Management
  [2] cosine=0.7261  Hypertension - Classification, Diagnosis, and Management
  [3] cosine=0.6669  Type 2 Diabetes Mellitus - Diagnosis and Glycaemic Management

ANSWER:
Stage 2 hypertension in adults is defined as having a systolic blood pressure of 140 or higher and/or a diastolic blood pressure of 90 or higher.

QUESTION: Which HbA1c value confirms a diagnosis of diabetes, and what range counts as prediabetes?

RETRIEVED (top 3):
  [1] cosine=0.7987  Type 2 Diabetes Mellitus - Diagnosis and Glycaemic Management
  [2] cosine=0.7249  Type 2 Diabetes Mellitus - Diagnosis and Glycaemic Management
  [3] cosine=0.6753  Type 2 Diabetes Mellitus - Diagnosis and Glycaemic Management

ANSWER:
An HbA1c value of 6.5% (or 48 mmol/mol) or higher confirms a diagnosis of type 2 diabetes mellitus. Prediabetes is defined by fas

### Retrieval evaluation

Retrieval is scored separately from generation, because it sets the ceiling: if the
right document is not in the top 3, no prompt can rescue the answer.

Two standard, easily explained metrics over the six labelled questions:

- **Hit@3** - fraction of questions whose expected document appears anywhere in the
  top 3.
- **MRR@3** - mean reciprocal rank: 1.0 if the correct document ranked first, 0.5 if
  second, 0.33 if third, 0 if absent. It rewards ranking the right document *first*,
  not merely including it.

Six labelled questions is a small sample and these numbers are not a benchmark result -
they are a sanity check that retrieval is working before we blame the generator.

In [28]:
def evaluate_retrieval(retriever: Retriever, questions: list[dict] = None, top_k: int = TOP_K) -> dict:
    """Hit@3 and MRR@3 over the labelled questions.

    Retrieval is scored on its own, before generation, because it sets the ceiling:
    if the right document is not in the top 3, no prompt can rescue the answer.
      Hit@3 - was the expected document retrieved anywhere in the top 3?
      MRR@3 - mean of 1/rank of the first correct hit, so ranking it first scores higher.
    """
    questions = questions or SAMPLE_QUESTIONS
    labelled = [q for q in questions if q["expected"]]
    hits = 0
    reciprocal_ranks = 0.0
    rows = []

    for item in labelled:
        results = retriever.retrieve_documents(item["question"], top_k=top_k)
        ranks = [r.rank for r in results if r.chunk.doc_id == item["expected"]]
        first_rank = min(ranks) if ranks else None
        hits += first_rank is not None
        reciprocal_ranks += 1.0 / first_rank if first_rank else 0.0
        rows.append({
            "question": item["question"],
            "expected": item["expected"],
            "first_correct_rank": first_rank,
            "top_score": round(results[0].score, 4),
        })

    n = len(labelled)
    return {"n": n, "hit@3": hits / n, "mrr@3": reciprocal_ranks / n, "rows": rows}

In [29]:
report = evaluate_retrieval(retriever)

print(f"{'rank':>5}  {'top cosine':>10}  expected document")
print("-" * 60)
for row in report["rows"]:
    rank = row["first_correct_rank"] or "MISS"
    print(f"{rank:>5}  {row['top_score']:>10.4f}  {row['expected']}")

print(f"\nHit@3 = {report['hit@3']:.2f}    MRR@3 = {report['mrr@3']:.2f}    (n={report['n']})")

# Confirm the pipeline returns exactly 3 results, as the task requires.
assert all(len(r["retrieved"]) == 3 for r in results), "expected exactly 3 retrieved docs"
print(f"\nAll {len(results)} questions returned exactly 3 retrieved documents.")

 rank  top cosine  expected document
------------------------------------------------------------
    1      0.8187  hypertension
    1      0.7987  type2_diabetes
    1      0.7759  malaria
    1      0.8698  anaphylaxis
    1      0.8168  pneumonia
    1      0.8370  tuberculosis

Hit@3 = 1.00    MRR@3 = 1.00    (n=6)

All 7 questions returned exactly 3 retrieved documents.


In [30]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [31]:
import shutil
from google.colab import files

shutil.make_archive("medical_corpus", "zip", "corpus")
files.download("medical_corpus.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Section 13 - Limitations

**This is an educational demonstration, not a clinical tool.** Its limits are real and
worth stating plainly.

**The knowledge base is small and not comprehensive.** 16 documents on 16 conditions.
Medicine covers thousands. Anything outside these topics simply cannot be answered, and
even within them the documents are condensed summaries, not the full guidelines.

**Answers are only as good as what was retrieved.** The generator never sees the corpus,
only the top 3 chunks. If retrieval misses the right passage, the answer will be wrong
or incomplete no matter how capable the model is.

**Retrieval always returns something.** A flat index returns its 3 nearest neighbours
whether or not they are relevant - as the out-of-scope question shows. The system relies
on the prompt for the model to notice the mismatch and say so, and a small model will
not always notice.

**The LLM can still make mistakes.** Even instructed to use only the context, a 1.5B
model can misread a threshold, merge two passages, drop a qualifier, or state something
with more confidence than the source did. Grounding reduces hallucination; it does not
eliminate it.

**Guidelines differ and change.** The corpus itself shows this - ACC/AHA and WHO use
different blood-pressure thresholds. Documents also go out of date, and nothing here
checks whether a source is current.

**No clinical context.** The system answers text questions from documents. It has no
patient history, comorbidities, allergies, pregnancy status, weight, renal function, or
local resistance patterns - all of which change real clinical decisions.

**Evaluation is minimal.** Hit@3 and MRR@3 over six questions written by the same person
who wrote the corpus. Enough to show retrieval works; nowhere near a benchmark. Answer
quality is judged by reading, not measured.

### Medical safety

This system **must not** be used for diagnosis, treatment decisions, dosing, or triage.
It is a demonstration of a retrieval architecture that happens to use medical text.
Medical information should be verified with qualified professionals and with the primary
guidelines named in each document's `source` field. In an emergency, contact emergency
services - do not consult a notebook.

---
## Section 14 - Conclusion

A complete RAG pipeline, built from open-source parts, running end to end:

**16 curated medical documents** -> cleaned -> **55 paragraph-based chunks** ->
**384-dimensional BGE embeddings** -> **FAISS exact cosine index** -> **top-3 retrieval**
-> **grounded prompt** -> **Qwen2.5-1.5B-Instruct** -> **an answer built only from the
retrieved context.**

Demonstrated on 7 questions - 6 answered from the corpus, 1 correctly identified as
outside it - with retrieval scored separately by Hit@3 and MRR@3.

The design goal was **correctness > simplicity > explainability > sophistication**. The
pipeline is dense retrieval and one generator, and every line of it can be explained.
Where a more advanced component was possible - hybrid retrieval, rank fusion, reranking,
similarity thresholds - it was left out because measurement showed the simple version
already answers these questions correctly. Complexity you cannot justify is complexity
you cannot debug.

---

## Why each component was used

| Component | Why it is here |
|---|---|
| **16-document curated corpus** | The task asks for 10-20 trusted medical references. Hand-authored from named WHO/CDC/NICE/specialist-society guidance so every answer is attributable and no unreliable web source is involved. |
| **Metadata (`id`, `title`, `source`, `category`)** | Required by the task, and it is what makes attribution possible - the `source` travels with each chunk into the prompt and out into the answer. |
| **Light cleaning** | Removes only what would genuinely distort an embedding - HTML tags, repeated spaces, extra blank lines. The corpus is already clean Markdown, so anything more would be effort spent on a problem that does not exist. |
| **Paragraph-based chunking (~180 words, 1-paragraph overlap)** | A whole document in one vector blurs its topics; a fixed word count slices bullet lists in half. Paragraphs are this corpus's natural units of meaning. Overlap keeps boundary facts readable. |
| **`BAAI/bge-small-en-v1.5` embeddings** | Open source, 33M parameters, runs on free CPU, and strong on MTEB retrieval for its size. Vector search matches "high blood pressure" to "hypertension", which keyword search cannot. |
| **Normalised vectors + query prefix** | Normalising makes inner product equal cosine similarity, so no extra step is needed at query time. The prefix is how BGE was trained to encode queries; omitting it measurably hurts retrieval. |
| **FAISS `IndexFlatIP`** | Required vector store. Flat = exact search, no approximation and no tuning parameters. At 55 vectors an approximate index would lose recall and save no measurable time. |
| **`top_k = 3`** | The task's requirement. Also a sound default: enough context to survive one weak match, short enough to keep the prompt focused. |
| **Grounded prompt with an explicit refusal sentence** | This is what actually enforces "answer only from the context". Giving the model an exact sentence for the no-answer case makes admitting a gap easier than inventing one. |
| **`Qwen2.5-1.5B-Instruct`** | Open weights (Apache-2.0), small enough for a free Colab runtime, instruction-tuned so it follows the grounding rule, 32k context. It reads the retrieved passages; it is not the source of the facts. |
| **Greedy decoding** | Reproducibility. The same question gives the same answer every run, which matters more than variety for medical text. |
| **Hit@3 / MRR@3** | Two lines of arithmetic that answer the one question worth asking before blaming the generator: did the right document actually get retrieved? |
| **One out-of-scope question** | The cheapest way to show the system admits what it does not know, rather than only demonstrating the happy path. |

